In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
response = gemini.chat.completions.create(model="gemini-2.5-flash",
messages=[{"role":"user", "content": "what is 2+2?"}])
print(response.choices[0].message.content)

2 + 2 = 4


In [30]:
links=fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

In [18]:
link_system_prompt = """You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}"""

In [3]:
    
def user_links_prompt(url):
    user_links_prompt =f"""
    Here is the list of links on the website {url}
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
    """
    links = fetch_website_links(url)
    user_links_prompt += "\n".join(links)
    return user_links_prompt


In [33]:
print(user_links_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
    https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/


In [4]:
def select_relevant_links(url):
    response= gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_links_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result= response.choices[0].message.content
    links = json.loads(result)
    return links

In [35]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services/offerings page',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services/offerings page',
   'url': 'https://edwarddonner.com/proficient/'}]}

In [22]:
def select_relevant_links(url):
    response= gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_links_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result= response.choices[0].message.content
    links = json.loads(result)
    print(f"found {len(links['links'])} relevant links ")
    return links

In [23]:
select_relevant_links("https://edwarddonner.com")

found 5 relevant links 


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'expertise page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'related product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [38]:
select_relevant_links("https://huggingface.co")

found 7 relevant links 


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand information', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'company blog', 'url': 'https://huggingface.co/blog'}]}

In [12]:
def fetch_webpage_relevant_link(url):
    content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    print(relevant_links)  # ← ADD THIS and tell me what it prints
    
    result = f"## Landing Page:\n\n{content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [83]:
print(select_relevant_links)

<function select_relevant_links at 0x0000022CD75B19E0>


In [35]:
def fetch_webpage_relevant_link(url):
    content=fetch_website_contents(url)
    relevant_links=select_relevant_links(url)
    
    print(f"DEBUG: relevant_links = {relevant_links}")
    print(f"DEBUG: Number of links = {len(relevant_links.get('links', []))}")
    
    result=f"## landing page:\n\n{content}\n\n## and relevant links:\n"
    
    if not relevant_links.get('links'):
        print("WARNING: No relevant links found")
        return result
    
    for link in relevant_links['links']:
        try:
            link_text = link['type'].upper()
            link_url = link["url"]
            result +=f"\n- [{link_text}]({link_url})\n"
            result +=fetch_website_contents(link_url)
        except Exception as e:
            print(f"ERROR fetching {link.get('url', 'unknown')}: {e}")
   
    return result

In [ ]:
print(fetch_webpage_relevant_link("https://huggingface.co"))

      

In [37]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
IMPORTANT: Include relevant links discovered about the company in your brochure output.
"""

In [39]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.
**IMPORTANT: Make sure to include all the relevant links from the "Relevant Links:" section in your output as clickable markdown links [link text](url).**\n\n
"""
    user_prompt += fetch_webpage_relevant_link(url)
    print(f"DEBUG: User prompt length before truncation: {len(user_prompt)}")
    user_prompt = user_prompt[:50_000]
    print(f"DEBUG: User prompt length after truncation: {len(user_prompt)}")
    print(f"DEBUG: Last 500 chars of prompt: {user_prompt[-500:]}")
    return user_prompt

In [40]:
def generate_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    brochure = response.choices[0].message.content
    display(Markdown(brochure))
   

In [ ]:
generate_brochure("hugging face", "https://huggingface.co")